#Monte Carlo Sensitivity Analysis – Terra/Luna Stages (2019–2021)
**Fracture Threshold Meter under Realistic Noise**
(5,000 iterations per stage | Gaussian noise: σ=1.0 on D1/G1/F2, σ=0.5 on H3/L2/C2, σ=0.3 on others | Collapse flag: >45%)
This analysis perturbs the consensus scores from each stage to test how stable the Fracture Meter remains under realistic uncertainty in scoring or measurement.
It shows whether small changes push the system across the hinge threshold — and how early the fragility signal appears.
Stage 1: Genesis (2019)
Stage 2: Anchor Launch (2020)
Stage 3: Parabolic Phase (2021)
Results summary:

Stage 1 → very low collapse risk (stable under noise)
Stage 2 → low collapse risk (mostly safe)
Stage 3 → high collapse risk (fragile — crosses hinge in most cases)

Monty Carlo Run Recalibration Terra stage 3

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    8.1, 7.4, 9.2, 2.4, 1.9, 5.2, 5.9, 4.0, 7.6, 7.1, 5.0, 7.8, 5.0, -5.0, 0.0, 6.3, 8.0,
    2.4, 2.5, 2.5, 2.3, 1.7, 2.7, 4.5, 2.2, 2.0, -4.0, -5.8, 1.8, 2.3, 6.4, -1.5, 0.0, -5.0, -1.8
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 51.2%
Median: 51.2%
95% CI: 45.5% – 56.9%
% of runs >45%: 98.2%
% of runs >50%: 65.8%
% of runs >55%: 10.0%
Collapse probability (>45% hinge): 98.2%


Stage 3 Version A – Shock: +1 to D1 (Exploitation tolerance)

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    8.1, 7.4, 9.2, 2.4, 1.9, 5.2, 5.9, 4.0, 7.6, 7.1, 5.0, 7.8, 5.0, -5.0, 0.0, 6.3, 8.0,
    2.4, 2.5, 2.5, 2.3, 1.7, 2.7, 4.5, 2.2, 2.0, -4.0, -5.8, 1.8, 2.3, 6.4, -1.5, 0.0, -5.0, -1.8
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # === SHOCK ADDED HERE ===
    perturbed[8] += 1.0          # +1 to D1 (exploitation tolerance)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 53.1%
Median: 53.2%
95% CI: 47.5% – 58.6%
% of runs >45%: 99.6%
% of runs >50%: 86.0%
% of runs >55%: 25.9%
Collapse probability (>45% hinge): 99.6%


Stage 3 Version B – Shock: -1 to G1 (Cheater detection)

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    8.1, 7.4, 9.2, 2.4, 1.9, 5.2, 5.9, 4.0, 7.6, 7.1, 5.0, 7.8, 5.0, -5.0, 0.0, 6.3, 8.0,
    2.4, 2.5, 2.5, 2.3, 1.7, 2.7, 4.5, 2.2, 2.0, -4.0, -5.8, 1.8, 2.3, 6.4, -1.5, 0.0, -5.0, -1.8
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # === SHOCK ADDED HERE ===
    perturbed[14] -= 1.0         # -1 to G1 (worse detection)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 53.2%
Median: 53.2%
95% CI: 47.5% – 58.9%
% of runs >45%: 99.6%
% of runs >50%: 86.0%
% of runs >55%: 26.8%
Collapse probability (>45% hinge): 99.6%


Stage 3 Combined shock version

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    8.1, 7.4, 9.2, 2.4, 1.9, 5.2, 5.9, 4.0, 7.6, 7.1, 5.0, 7.8, 5.0, -5.0, 0.0, 6.3, 8.0,
    2.4, 2.5, 2.5, 2.3, 1.7, 2.7, 4.5, 2.2, 2.0, -4.0, -5.8, 1.8, 2.3, 6.4, -1.5, 0.0, -5.0, -1.8
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # === COMBINED SHOCK ADDED HERE ===
    perturbed[8] += 1.0    # +1 to D1 (exploitation tolerance)
    perturbed[14] -= 1.0   # -1 to G1 (worse detection)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 55.1%
Median: 55.2%
95% CI: 49.5% – 60.6%
% of runs >45%: 100.0%
% of runs >50%: 96.1%
% of runs >55%: 52.9%
Collapse probability (>45% hinge): 100.0%


Monty Carlo Run stage 2

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    3.4, 4.5, 4.9, 2.2, 1.3, 3.3, 3.7, 2.8, 2.9, 3.9,
    1.3, 4.1, 2.8, -3.1, 0.1, 3.7, 4.7, 0.3, 0.7, 0.0,
    1.0, 1.0, 1.0, 2.7, 0.2, 0.6, -3.8, -3.8, 0.2, 1.5,
    2.8, -1.2, 0.0, -1.0, -1.7
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 41.0%
Median: 41.0%
95% CI: 35.2% – 46.7%
% of runs >45%: 8.6%
% of runs >50%: 0.1%
% of runs >55%: 0.0%
Collapse probability (>45% hinge): 8.6%


Monty Carlo run Stage 1

In [ ]:
import numpy as np

# === Paste the 35 consensus scores from terra_luna_stage3 here ===
# Order: A1 to M3 as in the CSV (35 values)
baseline = np.array([
    5.0, 5.8, 6.4, 2.1, 1.6, 4.1, 2.8, 1.8, 2.0, 4.1,
    1.1, 4.0, 3.0, -0.6, 1.1, 4.1, 5.9, 0.9, 1.6, 0.0,
    1.0, 1.0, 1.0, 2.4, 0.2, 0.6, -3.8, -3.8, 0.2, 1.5,
    2.8, -1.2, 0.0, -1.0, -1.7
])

# Indices of the six hinge metrics (0-based, from the 35-array)
hinge_indices = {
    'D1': 8,   # Exploitationism tolerance
    'G1': 14,  # Cheater detection
    'F2': 13,  # Error repair
    'H3': 19,  # Dissent suppression
    'L2': 30,  # Leadership cult
    'C2': 6    # Variation
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0   # noise std dev for D1, G1, F2
sigma_secondary = 0.5  # noise for H3, L2, C2
sigma_others = 0.3     # noise for all other metrics

np.random.seed(42)  # for reproducibility

fracture_values = []
collapse_flags = []

for i in range(n_iter):
    perturbed = baseline.copy()

    # Apply noise to all 35 metrics
    perturbed += np.random.normal(0, sigma_others, 35)

    # Apply stronger noise to hinge metrics
    for key, idx in hinge_indices.items():
        if key in ['D1', 'G1', 'F2']:
            perturbed[idx] += np.random.normal(0, sigma_main)
        else:
            perturbed[idx] += np.random.normal(0, sigma_secondary)

    # Clip to valid score range (-10 to +10)
    perturbed = np.clip(perturbed, -10, 10)

    # Recompute Fracture Meter (v1.0 formula for baseline comparison)
    d1 = perturbed[hinge_indices['D1']]
    g1 = perturbed[hinge_indices['G1']]
    f2 = perturbed[hinge_indices['F2']]
    h3 = perturbed[hinge_indices['H3']]
    l2 = perturbed[hinge_indices['L2']]
    c2 = perturbed[hinge_indices['C2']]

    term1 = d1 * 2
    term2 = (10 - g1) * 2
    term3 = 0.1 * (f2 + h3 + l2 + c2)
    fracture = 15 + term1 + term2 + term3

    fracture_values.append(fracture)

    # Collapse flag: using current >45% hinge as example (we can tune)
    collapse_flag = 1 if fracture > 45 else 0
    collapse_flags.append(collapse_flag)

# Summary statistics
mean_fracture = np.mean(fracture_values)
median_fracture = np.median(fracture_values)
ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
collapse_prob = np.mean(collapse_flags) * 100

print("Monte Carlo Results (5,000 iterations):")
print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
print(f"Median: {median_fracture:.1f}%")
print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
print(f"% of runs >45%: {pct_above_45:.1f}%")
print(f"% of runs >50%: {pct_above_50:.1f}%")
print(f"% of runs >55%: {pct_above_55:.1f}%")
print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

Monte Carlo Results (5,000 iterations):
Mean Fracture Meter: 37.3%
Median: 37.4%
95% CI: 31.6% – 43.1%
% of runs >45%: 0.4%
% of runs >50%: 0.0%
% of runs >55%: 0.0%
Collapse probability (>45% hinge): 0.4%


Complete Monty Carlo run + shocks Stage 1, 2, 3

In [ ]:
import numpy as np

# =======================
# BASELINE ARRAYS (35 metrics each)
# =======================

# Stage 1 (2019 Genesis) - from your earlier paste
baseline_stage1 = np.array([
    8.9, 8.4, 7.6, 2.8, 5.0, 7.8, 7.8, 5.2, 3.4, 7.0,
    2.3, 5.3, 5.3, 2.8, 1.5, 7.0, 8.5, -3.0, -3.0, -4.0,
    0.0, 0.0, 1.7, 4.0, 0.0, -3.0, 0.0, -4.0, 0.7, -1.0,
    3.0, 0.0, 0.0, -5.0, 0.0
])

# Stage 2 (2020 Anchor) - from your last paste
baseline_stage2 = np.array([
    5.0, 5.8, 6.4, 2.1, 1.6, 4.1, 2.8, 1.8, 2.0, 4.1,
    1.1, 4.0, 3.0, -0.6, 1.1, 4.1, 5.9, 0.9, 1.6, 0.0,
    1.0, 1.0, 1.0, 2.4, 0.2, 0.6, -3.8, -3.8, 0.2, 1.5,
    2.8, -1.2, 0.0, -1.0, -1.7
])

# Stage 3 (2021 Parabolic) - from QC'd consensus
baseline_stage3 = np.array([
    8.1, 7.4, 9.2, 2.4, 1.9, 5.2, 5.9, 4.0, 7.6, 7.1,
    5.0, 7.8, 5.0, -5.0, 0.0, 6.3, 8.0, 2.4, 2.5, 2.5,
    2.3, 1.7, 2.7, 4.5, 2.2, 2.0, -4.0, -5.8, 1.8, 2.3,
    6.4, -1.5, 0.0, -5.0, -1.8
])

# Hinge indices (0-based)
hinge_indices = {
    'D1': 8,
    'G1': 14,
    'F2': 13,
    'H3': 19,
    'L2': 30,
    'C2': 6
}

# Monte Carlo parameters
n_iter = 5000
sigma_main = 1.0
sigma_secondary = 0.5
sigma_others = 0.3

np.random.seed(42)

# =======================
# FUNCTION TO RUN MC ON A BASELINE
# =======================

def run_monte_carlo(baseline, label):
    fracture_values = []
    collapse_flags = []

    for i in range(n_iter):
        perturbed = baseline.copy()
        perturbed += np.random.normal(0, sigma_others, 35)

        for key, idx in hinge_indices.items():
            if key in ['D1', 'G1', 'F2']:
                perturbed[idx] += np.random.normal(0, sigma_main)
            else:
                perturbed[idx] += np.random.normal(0, sigma_secondary)

        perturbed = np.clip(perturbed, -10, 10)

        d1 = perturbed[hinge_indices['D1']]
        g1 = perturbed[hinge_indices['G1']]
        f2 = perturbed[hinge_indices['F2']]
        h3 = perturbed[hinge_indices['H3']]
        l2 = perturbed[hinge_indices['L2']]
        c2 = perturbed[hinge_indices['C2']]

        term1 = d1 * 2
        term2 = (10 - g1) * 2
        term3 = 0.1 * (f2 + h3 + l2 + c2)
        fracture = 15 + term1 + term2 + term3

        fracture_values.append(fracture)
        collapse_flag = 1 if fracture > 45 else 0
        collapse_flags.append(collapse_flag)

    mean_fracture = np.mean(fracture_values)
    ci_low, ci_high = np.percentile(fracture_values, [2.5, 97.5])
    pct_above_45 = np.mean(np.array(fracture_values) > 45) * 100
    pct_above_50 = np.mean(np.array(fracture_values) > 50) * 100
    pct_above_55 = np.mean(np.array(fracture_values) > 55) * 100
    collapse_prob = np.mean(collapse_flags) * 100

    print(f"\n{label} (5,000 iterations):")
    print(f"Mean Fracture Meter: {mean_fracture:.1f}%")
    print(f"95% CI: {ci_low:.1f}% – {ci_high:.1f}%")
    print(f"% >45%: {pct_above_45:.1f}%")
    print(f"% >50%: {pct_above_50:.1f}%")
    print(f"% >55%: {pct_above_55:.1f}%")
    print(f"Collapse probability (>45% hinge): {collapse_prob:.1f}%")

# =======================
# RUN FOR ALL STAGES
# =======================

run_monte_carlo(baseline_stage1, "Stage 1 (2019 Genesis)")
run_monte_carlo(baseline_stage2, "Stage 2 (2020 Anchor)")
run_monte_carlo(baseline_stage3, "Stage 3 (2021 Parabolic)")


Stage 1 (2019 Genesis) (5,000 iterations):
Mean Fracture Meter: 39.8%
95% CI: 34.0% – 45.5%
% >45%: 3.8%
% >50%: 0.0%
% >55%: 0.0%
Collapse probability (>45% hinge): 3.8%

Stage 2 (2020 Anchor) (5,000 iterations):
Mean Fracture Meter: 37.3%
95% CI: 31.3% – 43.1%
% >45%: 0.4%
% >50%: 0.0%
% >55%: 0.0%
Collapse probability (>45% hinge): 0.4%

Stage 3 (2021 Parabolic) (5,000 iterations):
Mean Fracture Meter: 51.2%
95% CI: 45.5% – 56.9%
% >45%: 98.3%
% >50%: 66.3%
% >55%: 9.3%
Collapse probability (>45% hinge): 98.3%
